# 🚀 CLSS — Train & Test trên Subset 25%

**Mục tiêu**: Train multi-view cooperative learning (SikuBERT + CRF) cho Sino-Nom Punctuation Restoration trên **25% dataset** để kiểm tra pipeline hoạt động đúng.

**Datasets trên Kaggle:**
- View 1 (non-RAG): `xunhunh/newdatasetnonrag`
- View 2 (RAG doc): `xunhunh/train1`
- Model: `xunhunh/sikubertlocal`

**Pipeline:**
1. Clone repo + cài dependencies
2. Tạo subset 25% từ cả 2 datasets
3. Patch config YAML
4. Train 10 epochs
5. Evaluate trên test set

## Cell 1 — Kiểm tra GPU & CUDA

In [ ]:
!nvidia-smi
!nvcc --version

## Cell 2 — Clone repo & cài dependencies

In [ ]:
!git clone https://github.com/thanhxuan217/CLSS.git
%cd /kaggle/working/CLSS
!pip install -r requirements.txt --upgrade -q

## Cell 3 — Kiểm tra dataset đầu vào

Xác nhận dữ liệu đã mount đúng trên Kaggle.

In [ ]:
import os

# ============================================================================
# CẤU HÌNH ĐƯỜNG DẪN — CHỈNH SỬA NẾU CẦN
# ============================================================================

# Dataset gốc (View 1 — non-RAG)
VIEW1_INPUT = "/kaggle/input/datasets/xunhunh/newdatasetnonrag/data/sino_nom_punct"

# Dataset doc (View 2 — RAG)
VIEW2_INPUT = "/kaggle/input/datasets/xunhunh/train1/data/sino_nom_punct_doc"

# SikuBERT model path
SIKUBERT_PATH = "/kaggle/input/models/xunhunh/sikubertlocal/pytorch/default/1/pretrained/sikubert"

# Output paths (writable)
VIEW1_SUBSET = "/kaggle/working/data/sino_nom_punct_subset"
VIEW2_SUBSET = "/kaggle/working/data/sino_nom_punct_doc_subset"
OUT_DIR = "/kaggle/working/output"

# ============================================================================
# KIỂM TRA
# ============================================================================
print("=" * 60)
print("Kiểm tra input datasets:")
print("=" * 60)

for name, path in [("View 1 (non-RAG)", VIEW1_INPUT), 
                    ("View 2 (RAG doc)", VIEW2_INPUT), 
                    ("SikuBERT model", SIKUBERT_PATH)]:
    exists = os.path.exists(path)
    status = "✅" if exists else "❌ KHÔNG TÌM THẤY"
    print(f"  {status} {name}: {path}")
    if exists and os.path.isdir(path):
        files = os.listdir(path)
        print(f"       Files: {files}")

print()

## Cell 4 — Tạo subset 25% từ cả 2 dataset

Script sẽ random sample 25% câu từ mỗi split (train/dev/test) của cả View 1 và View 2.

In [ ]:
# ============================================================================
# TẠO SUBSET 25% — VIEW 1 (non-RAG)
# ============================================================================
print("\n" + "=" * 60)
print("Tạo subset View 1 (non-RAG) — 25%")
print("=" * 60)

!python tools/create_subset.py \
    --input_dir {VIEW1_INPUT} \
    --output_dir {VIEW1_SUBSET} \
    --ratio 0.25 \
    --seed 42

# ============================================================================
# TẠO SUBSET 25% — VIEW 2 (RAG doc)
# ============================================================================
print("\n" + "=" * 60)
print("Tạo subset View 2 (RAG doc) — 25%")
print("=" * 60)

!python tools/create_subset.py \
    --input_dir {VIEW2_INPUT} \
    --output_dir {VIEW2_SUBSET} \
    --ratio 0.25 \
    --seed 42

## Cell 5 — Kiểm tra subset đã tạo

In [ ]:
import os

def count_conll_sentences(filepath):
    """Đếm số câu trong file CoNLL (phân cách bằng dòng trống)."""
    if not os.path.exists(filepath):
        return -1
    count = 0
    in_sentence = False
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line == '':
                if in_sentence:
                    count += 1
                    in_sentence = False
            else:
                in_sentence = True
    if in_sentence:
        count += 1
    return count

print("=" * 70)
print(f"{'Split':<10} {'View 1 Full':>12} {'View 1 Sub':>12} {'View 2 Full':>12} {'View 2 Sub':>12}")
print("-" * 70)

for split in ['train.txt', 'dev.txt', 'test.txt']:
    v1_full = count_conll_sentences(os.path.join(VIEW1_INPUT, split))
    v1_sub  = count_conll_sentences(os.path.join(VIEW1_SUBSET, split))
    v2_full = count_conll_sentences(os.path.join(VIEW2_INPUT, split))
    v2_sub  = count_conll_sentences(os.path.join(VIEW2_SUBSET, split))
    print(f"{split:<10} {v1_full:>12,} {v1_sub:>12,} {v2_full:>12,} {v2_sub:>12,}")

print("=" * 70)

## Cell 6 — Patch config YAML cho Kaggle

Cập nhật đường dẫn dataset, model, và output vào config file.

In [ ]:
import yaml

config_path = "config/sino_nom_doc_cl_subset.yaml"

with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# ============================================================================
# CẬP NHẬT ĐƯỜNG DẪN TUYỆT ĐỐI CHO KAGGLE
# ============================================================================

# Dataset paths → subset
config["punct"]["ColumnCorpus-SinoNomPunct"]["data_folder"] = VIEW1_SUBSET
config["punct"]["ColumnCorpus-SinoNomPunctDoc"]["data_folder"] = VIEW2_SUBSET

# Transformer model → local SikuBERT
config["embeddings"]["TransformerWordEmbeddings-0"]["model"] = SIKUBERT_PATH

# Tag dictionary → đường dẫn tuyệt đối
config["punct"]["tag_dictionary"] = "/kaggle/working/CLSS/resources/taggers/sino_nom_punct_tags.pkl"

# Output directory
config["target_dir"] = "/kaggle/working/output/"

print("\n📝 Config sau khi patch:")
print("=" * 60)
print(f"  View 1 data: {config['punct']['ColumnCorpus-SinoNomPunct']['data_folder']}")
print(f"  View 2 data: {config['punct']['ColumnCorpus-SinoNomPunctDoc']['data_folder']}")
print(f"  Transformer: {config['embeddings']['TransformerWordEmbeddings-0']['model']}")
print(f"  Epochs:      {config['train']['max_epochs']}")
print(f"  Batch size:  {config['train']['mini_batch_size']}")
print(f"  LR:          {config['train']['learning_rate']}")
print(f"  Output:      {config['target_dir']}")

# Ghi lại
with open(config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, allow_unicode=True, default_flow_style=False)

print("\n✅ Config đã được cập nhật!")

## Cell 7 — 🚀 TRAINING

Chạy training multi-view cooperative learning trên subset 25%. Khoảng 1-3 giờ trên T4 tùy kích thước dataset gốc.

In [ ]:
!python train.py \
    --config config/sino_nom_doc_cl_subset.yaml \
    --transformer_model {SIKUBERT_PATH} \
    --out_dir {OUT_DIR}

## Cell 8 — 📊 EVALUATION trên Test Set

Đánh giá model tốt nhất (best-model.pt) trên tập test của subset.

In [ ]:
!python train.py \
    --config config/sino_nom_doc_cl_subset.yaml \
    --test \
    --transformer_model {SIKUBERT_PATH} \
    --out_dir {OUT_DIR}

## Cell 9 — Kiểm tra kết quả

In [ ]:
import os

print("\n" + "=" * 60)
print("📁 Output files:")
print("=" * 60)

for root, dirs, files in os.walk(OUT_DIR):
    level = root.replace(OUT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in sorted(files):
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f'{subindent}{file} ({size_mb:.1f} MB)')

# Đọc training log nếu có
log_file = os.path.join(OUT_DIR, 'training.log')
if os.path.exists(log_file):
    print("\n" + "=" * 60)
    print("📋 Training log (last 30 lines):")
    print("=" * 60)
    with open(log_file, 'r') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())

## Cell 10 — (Tùy chọn) Đọc test results chi tiết

In [ ]:
# Tìm và đọc file kết quả test
import glob

result_files = glob.glob(os.path.join(OUT_DIR, '**', 'test.tsv'), recursive=True)
result_files += glob.glob(os.path.join(OUT_DIR, '**', '*result*'), recursive=True)

if result_files:
    for rf in result_files:
        print(f"\n📊 {rf}:")
        print("-" * 40)
        with open(rf, 'r') as f:
            print(f.read())
else:
    print("Kết quả test đã được in ở Cell 8 ở trên.")
    print("Tìm dòng 'MICRO_AVG' và 'MACRO_AVG' trong output.")